In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz, lfilter
from ipywidgets import HTML
from IPython.display import display

# ============================================================
# PADE APPROXIMATION FOR IIR FILTER DESIGN
# Theory, two examples, and numerical cross-check
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.pa-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.pa-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.pa-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14.5px;
    line-height:1.45;
    margin-bottom:7px;
}

.pa-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:7px;
    font-size:14px;
    line-height:1.45;
}

.pa-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
    margin-bottom:5px;
}

.pa-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.pa-col{
    flex:1;
    min-width:0;
}

.pa-note{
    background:#fff9e8;
    border:1px solid #d9c477;
}

.pa-ok{
    background:#eef7ee;
    border:1px solid #9cc79c;
}

.pa-code{
    font-family:Consolas,monospace;
    font-size:13px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="pa-root">

<div class="pa-header">
Padé Approximation for IIR Filter Design
</div>

<div class="pa-doc">

The Padé method approximates a desired impulse response
<b>h<sub>d</sub>[n]</b> with a rational IIR model

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
H(z) =
(β₀ + β₁z<sup>-1</sup> + ... + β<sub>M</sub>z<sup>-M</sup>) /
(1 + α₁z<sup>-1</sup> + ... + α<sub>N</sub>z<sup>-N</sup>).
</b>
</div>

Instead of solving nonlinear equations directly from the rational transfer
function, the impulse-response difference equation is used. The denominator
coefficients <b>αₖ</b> are determined first from samples for which the numerator
contribution is zero; the numerator coefficients <b>βₖ</b> are then calculated
from the first samples.

Padé therefore forces an exact match over a finite initial set of samples.
This notebook demonstrates exact recovery of a simple rational sequence
and applies the same procedure to the ideal low-pass with
<b>M = N = 5</b>.

</div>

</div>
"""))

# ============================================================
# HELPER FUNCTION
# ============================================================

def pade_from_samples(hd,M,N):

    X = np.zeros((N,N))
    rhs = np.zeros(N)

    for i in range(N):

        n = M+1+i

        rhs[i] = -hd[n]

        for k in range(1,N+1):

            X[i,k-1] = hd[n-k]

    alpha = np.linalg.solve(X,rhs)

    a = np.concatenate(([1.0],alpha))

    b = np.zeros(M+1)

    for n in range(M+1):

        value = hd[n]

        for k in range(1,min(N,n)+1):

            value += alpha[k-1]*hd[n-k]

        b[n] = value

    return b,a,X,rhs

# ============================================================
# EXAMPLE 1
#
# hd[n] = 4 (1/3)^n u[n]
# M = N = 1
# ============================================================

n1 = np.arange(12)

hd1 = 4.0*(1.0/3.0)**n1

M1 = 1
N1 = 1

b1,a1,X1,rhs1 = pade_from_samples(hd1,M1,N1)

h1 = lfilter(b1,a1,np.r_[1.0,np.zeros(len(n1)-1)])

E1 = np.sum((hd1-h1)**2)

# Book values

book_a1 = np.array([1.0,-1.0/3.0])
book_b1 = np.array([4.0,0.0])

diff_a1 = np.max(np.abs(a1-book_a1))
diff_b1 = np.max(np.abs(b1-book_b1))

# ============================================================
# EXAMPLE 2
#
# Ideal low-pass filter
# M = N = 5
#
# Use exactly the 11 samples printed in the text.
# ============================================================

hd2 = np.array([0.063661,0.000000,-0.106103,0.000000,0.318309,0.500000,0.318309,0.000000,-0.106103,0.000000,0.063661])

M2 = 5
N2 = 5

b2,a2,X2,rhs2 = pade_from_samples(hd2,M2,N2)

# ============================================================
# BOOK COEFFICIENTS FOR NUMERICAL CROSS-CHECK
# ============================================================

book_a2 = np.array([1.000000,-2.524810,3.675606,-3.483043,2.128893,-0.702628])

book_b2 = np.array([0.063661,-0.160731,0.127889,0.046155,0.063843,0.021161])

diff_a2 = np.max(np.abs(a2-book_a2))

diff_b2 = np.max(np.abs(b2-book_b2))

agreement_a2 = np.allclose(a2,book_a2,atol=2e-6,rtol=0)

agreement_b2 = np.allclose(b2,book_b2,atol=2e-6,rtol=0)

# ============================================================
# IMPULSE RESPONSE OF PADE FILTER
# ============================================================

L2 = 40

impulse = np.zeros(L2)

impulse[0] = 1.0

h2 = lfilter(b2,a2,impulse)

# Desired impulse-response expression used in the example
# hd[n] = sin[(n-5)pi/2] / [pi(n-5)]
# with the limiting value 1/2 at n = 5.

n2 = np.arange(L2)

hd2_long = np.zeros(L2)

for i,n in enumerate(n2):

    if n == 5:

        hd2_long[i] = 0.5

    else:

        hd2_long[i] = np.sin((n-5)*np.pi/2)/(np.pi*(n-5))

# ============================================================
# QUADRATIC ERROR
# ============================================================

error2 = hd2_long-h2

E2_11 = np.sum((hd2-h2[:11])**2)

E2_long = np.sum(error2**2)

# ============================================================
# POLES AND STABILITY
# ============================================================

poles2 = np.roots(a2)

max_pole_radius = np.max(np.abs(poles2))

stable2 = max_pole_radius < 1.0

# ============================================================
# FREQUENCY RESPONSE
# ============================================================

omega,H2 = freqz(b2,a2,worN=32768)

mag2 = np.abs(H2)

omega_norm = omega/np.pi

ideal_mag = np.where(omega <= np.pi/2,1.0,0.0)

# ============================================================
# DISPLAY — EXAMPLE 1
# ============================================================

display(HTML(f"""
<div class="pa-root">

<div class="pa-box">

<div class="pa-title">Example 1 — Exact recovery of a rational sequence</div>

<div class="pa-cols">

<div class="pa-col">

Desired impulse response:<br>

<b>
h<sub>d</sub>[n] = 4(1/3)<sup>n</sup>u[n]
</b>

<br><br>

Model order:
<b>M = 1, N = 1</b>

</div>

<div class="pa-col">

Calculated coefficients:<br>

α₁ = <b>{a1[1]:.9f}</b><br>

β₀ = <b>{b1[0]:.9f}</b><br>

β₁ = <b>{b1[1]:.3e}</b>

</div>

<div class="pa-col">

Quadratic error:<br>

<b>E = {E1:.3e}</b>

<br><br>

Maximum difference from book values:<br>

a: <b>{diff_a1:.3e}</b><br>
b: <b>{diff_b1:.3e}</b>

</div>

</div>

<div style="margin-top:7px;text-align:center;font-size:14.5px;">

<b>
H(z) = 4 / (1 - (1/3)z<sup>-1</sup>)
</b>

</div>

</div>

</div>
"""))

# ============================================================
# DISPLAY — EXAMPLE 2 RESULTS
# ============================================================

display(HTML(f"""
<div class="pa-root">

<div class="pa-box pa-note">

<div class="pa-title">Example 2 — Ideal low-pass approximation, M = N = 5</div>

The computation uses exactly the eleven desired impulse-response samples printed
in the numerical example.

<div style="margin-top:6px;" class="pa-code">
h<sub>d</sub>[n] =
[0.063661, 0, -0.106103, 0, 0.318309, 0.500000,
0.318309, 0, -0.106103, 0, 0.063661]
</div>

</div>

<div class="pa-box">

<div class="pa-title">Calculated Padé model</div>

<div class="pa-cols">

<div class="pa-col">

<b>Denominator coefficients</b><br>

a₀ = {a2[0]:.6f}<br>
a₁ = {a2[1]:.6f}<br>
a₂ = {a2[2]:.6f}<br>
a₃ = {a2[3]:.6f}<br>
a₄ = {a2[4]:.6f}<br>
a₅ = {a2[5]:.6f}

</div>

<div class="pa-col">

<b>Numerator coefficients</b><br>

β₀ = {b2[0]:.6f}<br>
β₁ = {b2[1]:.6f}<br>
β₂ = {b2[2]:.6f}<br>
β₃ = {b2[3]:.6f}<br>
β₄ = {b2[4]:.6f}<br>
β₅ = {b2[5]:.6f}

</div>

<div class="pa-col">

<b>Model diagnostics</b><br>

max |pole| = <b>{max_pole_radius:.6f}</b><br>

Stable = <b>{"YES" if stable2 else "NO"}</b><br><br>

Error over first 11 samples:<br>
<b>{E2_11:.6e}</b><br>

Error over 40 samples:<br>
<b>{E2_long:.6f}</b>

</div>

</div>

</div>

</div>
"""))

# ============================================================
# TRANSFER FUNCTION
# ============================================================

display(HTML(f"""
<div class="pa-root">

<div class="pa-box pa-note">

<div class="pa-title">Transfer function</div>

<div style="font-size:14px;line-height:1.65;">

H(z) =
<b>
({b2[0]:.6f}
{b2[1]:+.6f}z<sup>-1</sup>
{b2[2]:+.6f}z<sup>-2</sup>
{b2[3]:+.6f}z<sup>-3</sup>
{b2[4]:+.6f}z<sup>-4</sup>
{b2[5]:+.6f}z<sup>-5</sup>) /
</b>

<br>

<div style="padding-left:48px;">

<b>
(1
{a2[1]:+.6f}z<sup>-1</sup>
{a2[2]:+.6f}z<sup>-2</sup>
{a2[3]:+.6f}z<sup>-3</sup>
{a2[4]:+.6f}z<sup>-4</sup>
{a2[5]:+.6f}z<sup>-5</sup>)
</b>

</div>

</div>

</div>

</div>
"""))

# ============================================================
# NUMERICAL CROSS-CHECK WITH BOOK
# ============================================================

display(HTML(f"""
<div class="pa-root">

<div class="pa-box pa-ok">

<div class="pa-title">Numerical cross-check with the textbook example</div>

<div class="pa-cols">

<div class="pa-col">

<b>Book denominator</b><br>

<div class="pa-code">
[1.000000,<br>
-2.524810,<br>
3.675606,<br>
-3.483043,<br>
2.128893,<br>
-0.702628]
</div>

</div>

<div class="pa-col">

<b>Notebook denominator</b><br>

<div class="pa-code">
[{a2[0]:.6f},<br>
{a2[1]:.6f},<br>
{a2[2]:.6f},<br>
{a2[3]:.6f},<br>
{a2[4]:.6f},<br>
{a2[5]:.6f}]
</div>

</div>

<div class="pa-col">

<b>Book numerator</b><br>

<div class="pa-code">
[0.063661,<br>
-0.160731,<br>
0.127889,<br>
0.046155,<br>
0.063843,<br>
0.021161]
</div>

</div>

<div class="pa-col">

<b>Notebook numerator</b><br>

<div class="pa-code">
[{b2[0]:.6f},<br>
{b2[1]:.6f},<br>
{b2[2]:.6f},<br>
{b2[3]:.6f},<br>
{b2[4]:.6f},<br>
{b2[5]:.6f}]
</div>

</div>

</div>

<div style="margin-top:8px;text-align:center;">

Maximum denominator difference:
<b>{diff_a2:.3e}</b>

&nbsp;&nbsp;&nbsp;

Maximum numerator difference:
<b>{diff_b2:.3e}</b>

<br>

<b>
{"RESULT: The notebook reproduces the textbook coefficients within rounding precision." if agreement_a2 and agreement_b2 else "RESULT: A discrepancy exists and should be inspected."}
</b>

</div>

</div>

</div>
"""))

# ============================================================
# FIGURE — 2 x 2 GRID
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(9.0,6.4))

ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# 1. EXAMPLE 1 — DESIRED VS EXACT PADE RESPONSE
# ============================================================

marker_hd1,stem_hd1,base_hd1 = ax1.stem(n1,hd1,linefmt='C0-',markerfmt='C0o',basefmt=' ')

marker_h1,stem_h1,base_h1 = ax1.stem(n1,h1,linefmt='r--',markerfmt='ro',basefmt=' ')

plt.setp(stem_hd1,linewidth=1.1)

plt.setp(stem_h1,linewidth=1.0)

marker_hd1.set_markersize(4.5)

marker_h1.set_markersize(3.5)

marker_hd1.set_label(r'Desired $h_d[n]$')

marker_h1.set_label(r'Padé $h[n]$')

ax1.axhline(0,color='black',linewidth=0.8)

ax1.set_xlim(-0.5,11.5)

ax1.set_ylim(-0.1,4.3)

ax1.set_title('Example 1 — Exact Padé Recovery')

ax1.set_xlabel('Sample index n')

ax1.set_ylabel('Amplitude')

ax1.grid(True,linestyle=':',alpha=0.25)

ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# 2. EXAMPLE 2 — INITIAL IMPULSE-RESPONSE MATCH
# ============================================================

n_short = np.arange(11)

marker_hd2,stem_hd2,base_hd2 = ax2.stem(n_short,hd2,linefmt='C0-',markerfmt='C0o',basefmt=' ')

marker_h2,stem_h2,base_h2 = ax2.stem(n_short,h2[:11],linefmt='r--',markerfmt='ro',basefmt=' ')

plt.setp(stem_hd2,linewidth=1.1)

plt.setp(stem_h2,linewidth=1.0)

marker_hd2.set_markersize(4.5)

marker_h2.set_markersize(3.5)

marker_hd2.set_label(r'Desired $h_d[n]$')

marker_h2.set_label(r'Padé $h[n]$')

ax2.axhline(0,color='black',linewidth=0.8)

ax2.set_xlim(-0.5,10.5)

ax2.set_ylim(-0.2,0.58)

ax2.set_title('Example 2 — First 11 Samples')

ax2.set_xlabel('Sample index n')

ax2.set_ylabel('Amplitude')

ax2.grid(True,linestyle=':',alpha=0.25)

ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# 3. EXAMPLE 2 — LONGER IMPULSE RESPONSE
# ============================================================

ax3.plot(n2,hd2_long,color='black',linewidth=1.2,label=r'Desired $h_d[n]$')

ax3.plot(n2,h2,color='red',linewidth=1.3,label=r'Padé $h[n]$')

ax3.axhline(0,color='black',linewidth=0.8)

ax3.axvline(10,linestyle='--',linewidth=1.0,label='End of fitted samples')

ax3.set_xlim(0,L2-1)

ax3.set_ylim(-0.8,0.8)

ax3.set_title('What Happens Beyond the Matched Samples?')

ax3.set_xlabel('Sample index n')

ax3.set_ylabel('Amplitude')

ax3.grid(True,linestyle=':',alpha=0.25)

ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# 4. EXAMPLE 2 — MAGNITUDE RESPONSE
# ============================================================

ax4.plot(omega_norm,ideal_mag,color='black',linewidth=1.2,label='Ideal low-pass')

ax4.plot(omega_norm,mag2,color='red',linewidth=1.4,label='Padé approximation')

ax4.axvline(0.5,linestyle='--',linewidth=1.0,label=r'$\omega_c=\pi/2$')

ax4.set_xlim(0,1)

ax4.set_ylim(0,3.0)

ax4.set_title('Ideal vs Padé Magnitude Response')

ax4.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax4.set_ylabel(r'$|H(e^{j\omega})|$')

ax4.grid(True,linestyle=':',alpha=0.25)

ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# LAYOUT
# ============================================================

plt.subplots_adjust(left=0.08,right=0.98,top=0.94,bottom=0.13,wspace=0.28,hspace=0.56)

# ============================================================
# DISPLAY
# ============================================================

display(fig.canvas)